# 2026 FIFA World Cup Simulation and Team Rating Analysis

This notebook simulates the 2026 FIFA World Cup using actual group data. It includes:
*   **Team Rating Systems:** Massey, Keener, and Markov.
*   **Simulation Phases:** Group stage advancement and direct elimination knockout rounds.
*   **Objective:** Predict match outcomes and determine a tournament winner.

### 1: Historical data on International Matches - Download + Read Data

The dataset `results.csv` is obtained from the 'International Football Results from 1872 to 2026' Kaggle dataset, which compiles international football match results.

In [1]:
import pandas as pd
import numpy as np

In [2]:
HIST_DATA = "https://raw.githubusercontent.com/ale66/learn-datascience/main/week-9/Ranking_world_cup/results.csv"

In [3]:
df = pd.read_csv(HIST_DATA)

df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [4]:

df.tail(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49467,2026-06-26,Cape Verde,Saudi Arabia,NaN,NaN,FIFA World Cup,Houston,United States,True
49468,2026-06-26,Uruguay,Spain,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True
49469,2026-06-26,Norway,France,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49470,2026-06-26,Senegal,Iraq,NaN,NaN,FIFA World Cup,Toronto,Canada,True
49471,2026-06-27,Algeria,Austria,NaN,NaN,FIFA World Cup,Kansas City,United States,True
49472,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True
49473,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49474,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49475,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49476,2026-06-27,Croatia,Ghana,NaN,NaN,FIFA World Cup,Philadelphia,United States,True


Take historical data of the last 8 years, which contains two previous world cups

In [5]:
df = df[df["date"] >= "2018-01-01"]

In [6]:
print(df.shape)

df.head()

(8180, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
41297,2018-01-02,Iraq,United Arab Emirates,0.0,0.0,Gulf Cup,Kuwait City,Kuwait,True
41298,2018-01-02,Oman,Bahrain,1.0,0.0,Gulf Cup,Kuwait City,Kuwait,True
41299,2018-01-05,Oman,United Arab Emirates,0.0,0.0,Gulf Cup,Kuwait City,Kuwait,True
41300,2018-01-07,Estonia,Sweden,1.0,1.0,Friendly,Abu Dhabi,United Arab Emirates,True
41301,2018-01-11,Denmark,Sweden,0.0,1.0,Friendly,Abu Dhabi,United Arab Emirates,True


## 2026 FIFA World Cup - group stage

This section fetches the 2026 FIFA World Cup group stage data from a public JSON file.

It then processes this data to extract the teams participating in each group, which will be used for the simulation.

In [7]:
import requests

In [8]:
URL = "https://raw.githubusercontent.com/openfootball/worldcup.json/master/2026/worldcup.json"


In [9]:
data = requests.get(URL).json()

data


{'name': 'World Cup 2026',
 'matches': [{'round': 'Matchday 1',
   'date': '2026-06-11',
   'time': '13:00 UTC-6',
   'team1': 'Mexico',
   'team2': 'South Africa',
   'score': {'ft': [2, 0], 'ht': [1, 0]},
   'goals1': [{'name': 'Julián Quiñones', 'minute': '9'},
    {'name': 'Raúl Jiménez', 'minute': '67'}],
   'goals2': [],
   'group': 'Group A',
   'ground': 'Mexico City'},
  {'round': 'Matchday 1',
   'date': '2026-06-11',
   'time': '20:00 UTC-6',
   'team1': 'South Korea',
   'team2': 'Czech Republic',
   'score': {'ft': [2, 1], 'ht': [0, 0]},
   'goals1': [{'name': 'Hwang In-Beom', 'minute': '67'},
    {'name': 'Oh Hyeon-Gyu', 'minute': '80'}],
   'goals2': [{'name': 'Ladislav Krejcí', 'minute': '59'}],
   'group': 'Group A',
   'ground': 'Guadalajara (Zapopan)'},
  {'round': 'Matchday 8',
   'date': '2026-06-18',
   'time': '12:00 UTC-4',
   'team1': 'Czech Republic',
   'team2': 'South Africa',
   'score': {'ft': [1, 1], 'ht': [1, 0]},
   'goals1': [{'name': 'Michal Sadílek',

extract teams and their respective groups

In [10]:
groups = {}

teams = set()

for match in data["matches"]:

    team1 = match.get("team1")
    team2 = match.get("team2")
    group = match.get("group")

    # skip knockout placeholders like W101 etc.
    if not team1 or not team2:
        continue

    # only group stage matches
    if group is None:
        continue

    if "Group" not in group:
        continue

    if group not in groups:
        groups[group] = set()

    groups[group].add(team1)
    groups[group].add(team2)

    teams.add(team1)
    teams.add(team2)


Prepare groups for visualisation.

In [11]:
groups = {k: sorted(list(v)) for k, v in groups.items()}

teams = sorted(list(teams))

sanity check

In [12]:
print("number of teams:", len(teams))
print("number of groups:", len(groups))

for g, t in groups.items():
    print(g, t)

number of teams: 48
number of groups: 12
Group A ['Czech Republic', 'Mexico', 'South Africa', 'South Korea']
Group B ['Bosnia & Herzegovina', 'Canada', 'Qatar', 'Switzerland']
Group C ['Brazil', 'Haiti', 'Morocco', 'Scotland']
Group D ['Australia', 'Paraguay', 'Turkey', 'USA']
Group E ['Curaçao', 'Ecuador', 'Germany', 'Ivory Coast']
Group F ['Japan', 'Netherlands', 'Sweden', 'Tunisia']
Group G ['Belgium', 'Egypt', 'Iran', 'New Zealand']
Group H ['Cape Verde', 'Saudi Arabia', 'Spain', 'Uruguay']
Group I ['France', 'Iraq', 'Norway', 'Senegal']
Group J ['Algeria', 'Argentina', 'Austria', 'Jordan']
Group K ['Colombia', 'DR Congo', 'Portugal', 'Uzbekistan']
Group L ['Croatia', 'England', 'Ghana', 'Panama']


Pick up the corresponding dataset

In [13]:
df_48 = df[
    (df["home_team"].isin(teams)) &
    (df["away_team"].isin(teams))
].copy()

## 2: Massey Ratings (Linear Algebra Version)


Core Formula: $\overline{M}\mathbf{r}=\mathbf{p}$

where

*   $M$ = simple match matrix, always the same;
*   $\overline{M}$ = Massey's *mutilated* matrix;
*   $\mathbf{r}$ = ratings vector, to be solved, and
*   $\mathbf{p}$ = goal difference vector, known.

#### Construct the simple M matrix

1. Teams

In [14]:
n = len(teams)

team_index = {t:i for i,t in enumerate(teams)}


2. Fill Matrix

In [15]:
M = np.zeros((n, n))
p = np.zeros(n)

for _, row in df_48.iterrows():
    home = team_index[row["home_team"]]
    away = team_index[row["away_team"]]

    hs = row["home_score"]
    as_ = row["away_score"]

    if np.isnan(hs) or np.isnan(as_):
        continue

    diff = hs - as_

    # Massey matrix
    M[home, home] += 1
    M[away, away] += 1
    M[home, away] -= 1
    M[away, home] -= 1

    p[home] += diff
    p[away] -= diff

##### Now construct Massey's mutilated matrix $\overline{M}$

In [16]:
M[-1, :] = 1
p[-1] = 0

1. Solve  $\overline{M}\mathbf{r}=\mathbf{p}$ by finding the least-squares solution, thanks to `numpy.linalg.lstsq`.

In [17]:
ratings_massey = np.linalg.lstsq(M, p, rcond=None)[0]

5. Output

In [18]:
massey_df = pd.DataFrame({
    "team": teams,
    "massey": ratings_massey
})

massey_df = massey_df.sort_values("massey", ascending=False)
massey_df.reset_index(drop=True, inplace=True)

massey_df.head(10)

,team,massey
0,France,1.400019
1,Spain,1.291098
2,Brazil,1.282712
3,Portugal,1.250632
4,Argentina,1.236670
5,Belgium,1.198854
6,Netherlands,1.044432
7,England,0.992939
8,Austria,0.862708
9,Colombia,0.710640


## 3: Keener Ratings (Perron vector)

Core Idea: solve the eigenvalue problem

$A\mathbf{r} = \lambda \mathbf{r}$

to find the dominant (Perron) vector, which will represent the team ratings.

If $A$ is irreducible and non-negative, the Perron-Frobenius theorem guarantees a unique positive eigenvector corresponding to the largest eigenvalue $\lambda$.


In [19]:
A = np.zeros((n, n))


In [20]:
# Filter out rows with NaN scores from df_48 for Keener ratings calculation
df_48_keener = df_48.dropna(subset=["home_score", "away_score"]).copy()

for _, row in df_48_keener.iterrows():
    i = team_index[row["home_team"]]
    j = team_index[row["away_team"]]

    A[i, j] += row["home_score"] + 1

    A[j, i] += row["away_score"] + 1



Normalisation

In [21]:
col_sum = A.sum(axis=0)

# Avoid division by zero for teams that might not have played any match
# Replace 0 with 1 in col_sum where it's 0 to prevent NaN/Inf in A after division
col_sum[col_sum == 0] = 1

A = A / col_sum


##### Calculate Perron's eigenvector

In [22]:
# Ensure A is not all zeros or contains NaNs
# Also handle cases where A might contain inf
if not np.all(A == 0) and not np.any(np.isnan(A)) and not np.any(np.isinf(A)):

    eigvals, eigvecs = np.linalg.eig(A)

    idx = np.argmax(eigvals.real)

    keener_ratings = eigvecs[:, idx].real

    # Key: Standardize (non-negative + normalize)
    keener_ratings = np.abs(keener_ratings)
    keener_ratings = keener_ratings / keener_ratings.sum()

    keener_df = pd.DataFrame({
        "team": teams,
        "rating": keener_ratings
    }).sort_values("rating", ascending=False)

    keener_df.reset_index(drop=True, inplace=True)

    print(keener_df.head(10))
else:
    print("Matrix A is invalid (all zeros, contains NaN, or Inf values). Cannot compute Keener ratings.")

          team    rating
0        Spain  0.046338
1       France  0.044326
2       Brazil  0.042489
3      Croatia  0.039126
4    Argentina  0.037802
5     Colombia  0.035381
6  Netherlands  0.034996
7     Portugal  0.034751
8      Germany  0.033766
9        Japan  0.032498


## 4. Markov Ratings

This section calculates Markov ratings using a Monte Carlo simulation. This approach constructs transition probabilities based on goals scored and conceded, rather than traditional win/loss transition matrices. We will build a `match_grid` representing match scores between teams and derive an `S_dataframe` (a transition probability matrix based on goals conceded). A Monte Carlo simulation will then perform random walks on this transition matrix, counting team visits to determine their final ratings.

In [23]:
match_grid = pd.DataFrame(index=teams, columns=teams, dtype=str)

for _, row in df_48.iterrows():
    home_team = row["home_team"]
    away_team = row["away_team"]
    home_score = row["home_score"]
    away_score = row["away_score"]

    if pd.notna(home_score) and pd.notna(away_score):
        # Store the score as a string 'home_score-away_score'
        match_grid.loc[home_team, away_team] = f"{int(home_score)}-{int(away_score)}"

print("Partial match_grid (first 5 rows and columns):")
display(match_grid.head())

Partial match_grid (first 5 rows and columns):


,Algeria,Argentina,Australia,Austria,Belgium,Bosnia & Herzegovina,Brazil,Canada,Cape Verde,Colombia,...,South Africa,South Korea,Spain,Sweden,Switzerland,Tunisia,Turkey,USA,Uruguay,Uzbekistan
Algeria,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5-1,3-0,...,3-3,NaN,NaN,NaN,NaN,1-1,NaN,NaN,0-0,NaN
Argentina,3-0,NaN,2-0,2-0,NaN,NaN,4-1,2-0,NaN,1-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0-2,NaN
Australia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0-0,...,NaN,1-2,NaN,NaN,1-1,NaN,2-0,NaN,NaN,1-1
Austria,NaN,NaN,NaN,NaN,2-3,NaN,0-3,NaN,NaN,NaN,...,NaN,1-0,NaN,2-0,NaN,1-0,1-2,NaN,NaN,NaN
Belgium,NaN,NaN,NaN,1-1,NaN,NaN,NaN,1-0,NaN,NaN,...,NaN,NaN,NaN,1-1,2-1,5-0,NaN,NaN,NaN,NaN


In [24]:
# Parse score and get goals conceded at home
home_goals_ij = lambda score: int(score.split("-")[1]) if pd.notna(score) else 0

all_home_goals_ij = match_grid.map(home_goals_ij)

# Parse score and get goals conceded away
away_goals_ij = lambda score: int(score.split("-")[0]) if pd.notna(score) else 0

all_away_goals_ij = match_grid.T.map(away_goals_ij)

# Sum goals conceded
V_dataframe = all_home_goals_ij + all_away_goals_ij  # row_sums: Sum of goals each team conceded

# Ensure each row in V_dataframe has at least one non-zero value to prevent division by zero.
# If a team has conceded no goals, its row_sum could be 0. We set such row_sums to 1 to avoid errors.
row_sums = V_dataframe.sum(axis=1)
row_sums[row_sums == 0] = 1

# Create S_dataframe (normalized transition matrix)
S_dataframe = V_dataframe.div(row_sums, axis=0)

# Dictionary with teams as keys and lists of probabilities as values.
# Each list represents a probability of moving from the current team
# to another team in the league (fair-weather fan logic).
transit_dict = S_dataframe.T.to_dict(orient="list")

# Dictionary with teams as keys and number of visits as values.
counter_dict = {team: 0 for team in teams}

print("Transition probability matrix S_dataframe (partial):")
display(S_dataframe.head())

Transition probability matrix S_dataframe (partial):


,Algeria,Argentina,Australia,Austria,Belgium,Bosnia & Herzegovina,Brazil,Canada,Cape Verde,Colombia,...,South Africa,South Korea,Spain,Sweden,Switzerland,Tunisia,Turkey,USA,Uruguay,Uzbekistan
Algeria,0.0,0.100000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.033333,0.000000,...,0.1,0.000000,0.000000,0.133333,0.000000,0.033333,0.000000,0.0,0.000000,0.000000
Argentina,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.034483,0.0,0.000000,0.103448,...,0.0,0.000000,0.206897,0.000000,0.000000,0.000000,0.000000,0.0,0.068966,0.000000
Australia,0.0,0.074074,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.111111,...,0.0,0.111111,0.000000,0.000000,0.037037,0.000000,0.000000,0.0,0.000000,0.037037
Austria,0.0,0.064516,0.0,0.000000,0.129032,0.0,0.096774,0.0,0.000000,0.000000,...,0.0,0.000000,0.000000,0.032258,0.032258,0.000000,0.064516,0.0,0.000000,0.000000
Belgium,0.0,0.000000,0.0,0.088235,0.000000,0.0,0.029412,0.0,0.000000,0.000000,...,0.0,0.000000,0.000000,0.029412,0.176471,0.000000,0.000000,0.0,0.000000,0.000000


In [25]:
# Number of iterations for the simulation
N = 100_000

# Initialize the process by randomly selecting a starting team
curr_team = np.random.choice(teams)
counter_dict[curr_team] += 1

# Run the simulation for N iterations
for i in range(N):
    # Get the transition probabilities for the current team
    # Ensure the sum of probabilities in transit_dict is 1 to avoid errors with np.random.choice
    probs = transit_dict[curr_team]

    # If the probability list is empty or all probabilities are zero, randomly select the next team
    if not probs or np.sum(probs) == 0:
        curr_team = np.random.choice(teams)
    else:
        # Normalize probabilities if their sum is not 1
        prob_sum = np.sum(probs)
        if prob_sum != 1:
            probs = np.array(probs) / prob_sum
        curr_team = np.random.choice(teams, p=probs)

    counter_dict[curr_team] += 1

# Calculate ratings as long-term visit frequencies
ratings = [count / (N + 1) for count in counter_dict.values()]

# print(f"Markov ratings obtained after {N} simulation runs (partial):\n{ratings[:10]}")

In [26]:
# Convert simulation results to a DataFrame.
markov_df = (
    pd.DataFrame({"team": teams, "rating": ratings})
    .sort_values(by="rating", ascending=False)
    .reset_index(drop=True)
)

print("New Markov ratings (markov_df):")
display(markov_df.head(10))

New Markov ratings (markov_df):


,team,rating
0,Spain,0.058469
1,France,0.058149
2,Brazil,0.041870
3,Argentina,0.039560
4,Netherlands,0.037810
5,Switzerland,0.037100
6,Japan,0.037000
7,Colombia,0.035960
8,Portugal,0.035750
9,Belgium,0.035240


### 5. Putting it All Together: 2026 World Cup Simulation

Create a single function `simulate_world_cup` that can be used with any rating system (Massey, Keener, Markov). This function will handle:

1.  **Group Stage Advancement:** Identifying the top two teams from each group and the best eight third-placed teams.
2.  **Knockout Rounds:** Simulating matches based on the provided ratings until a single winner is determined.

#### Simulating Individual Matches

In [27]:
import random

def simulate_match(team1, team2, rating_map):
    rating1 = rating_map.get(team1)
    rating2 = rating_map.get(team2)

    if rating1 is None or rating2 is None:
        raise ValueError(f"Missing rating for {team1} or {team2}")

    # Calculate win probability for team1
    total_strength = rating1 + rating2
    if total_strength == 0: # Handle case where both ratings are 0 to avoid division by zero
        prob_team1_wins = 0.5
    else:
        prob_team1_wins = rating1 / total_strength

    # Determine winner based on probability
    if random.random() < prob_team1_wins:
        return team1
    else:
        return team2

#### Group Stage and Third-Place Advancement

In [28]:
def process_groups(ratings_df, rating_col, groups):
    advancing = {}
    third_place = []

    for g, teams in groups.items():
        df = ratings_df[ratings_df['team'].isin(teams)] \
             .sort_values(rating_col, ascending=False)

        advancing[g] = df['team'].head(2).tolist()

        if len(df) >= 3:
            third_place.append((df.iloc[2]['team'], df.iloc[2][rating_col], g))

    third_place = sorted(third_place, key=lambda x: x[1], reverse=True)[:8]
    third_place = [t[0] for t in third_place]

    return advancing, third_place

#### Knockout Round Progression

In [29]:
def play_round(teams, ratings_df, rating_col, round_name=""):
    if len(teams) % 2:
        winners, pairs = [teams.pop(0)], zip(teams[::2], teams[1::2])
    else:
        winners, pairs = [], zip(teams[::2], teams[1::2])

    matches = []

    # Create a rating_map from the ratings_df for efficient lookup
    rating_map = ratings_df.set_index('team')[rating_col].to_dict()

    for t1, t2 in pairs:
        w = simulate_match(t1, t2, rating_map)
        winners.append(w)
        matches.append({"team1": t1, "team2": t2, "winner": w})

    return winners, {"round": round_name, "matches": matches}

#### `simulate_world_cup` Function: Running the Full Tournament Simulation

This is the main simulation function that orchestrates the entire World Cup. It first processes the group stage to determine the initial set of advancing teams. Then, it iteratively runs knockout rounds until a single champion is determined. It also collects the results of each round in a bracket for later display.

In [30]:
def simulate_world_cup(ratings_df, rating_column_name, groups, simulation_name, initial_round_size=32):
    print(f"\n--- World Cup Simulation ({simulation_name}) ---")

    advancing, third_place = process_groups(ratings_df, rating_column_name, groups)

    current = [t for v in advancing.values() for t in v] + third_place
    random.shuffle(current)

    bracket = []
    round_size = initial_round_size

    while len(current) > 1:
        current, round_result = play_round(
            current,
            ratings_df,
            rating_column_name,
            f"Round of {round_size}"
        )
        bracket.append(round_result)
        round_size //= 2

    print(f"\n🏆 Winner ({simulation_name}): {current[0]}")

    return current[0], bracket

In [31]:
def print_bracket(bracket):
    for r in bracket:
        print(f"\n=== {r['round']} ===")
        for m in r['matches']:
            print(f"{m['team1']} vs {m['team2']} → {m['winner']}")

In [32]:
massey_winner, bracket = simulate_world_cup(massey_df, 'massey', groups, 'Massey')

print_bracket(bracket)


--- World Cup Simulation (Massey) ---

🏆 Winner (Massey): Argentina

=== Round of 32 ===
Portugal vs Bosnia & Herzegovina → Portugal
Argentina vs Egypt → Argentina
Algeria vs Japan → Japan
Ghana vs France → France
Netherlands vs Switzerland → Netherlands
Germany vs South Korea → Germany
Spain vs Belgium → Belgium
Canada vs Uruguay → Uruguay
USA vs Brazil → Brazil
Morocco vs Paraguay → Morocco
Scotland vs Colombia → Colombia
Mexico vs Turkey → Mexico
Ivory Coast vs Austria → Austria
Senegal vs Sweden → Sweden
Croatia vs England → England
Norway vs Ecuador → Norway

=== Round of 16 ===
Portugal vs Argentina → Argentina
Japan vs France → France
Netherlands vs Germany → Netherlands
Belgium vs Uruguay → Belgium
Brazil vs Morocco → Brazil
Colombia vs Mexico → Colombia
Austria vs Sweden → Austria
England vs Norway → England

=== Round of 8 ===
Argentina vs France → Argentina
Netherlands vs Belgium → Belgium
Brazil vs Colombia → Brazil
Austria vs England → England

=== Round of 4 ===
Argentin

In [33]:
keener_winner, bracket = simulate_world_cup(keener_df, 'rating', groups, 'Keener')

print_bracket(bracket)


--- World Cup Simulation (Keener) ---

🏆 Winner (Keener): Mexico

=== Round of 32 ===
Brazil vs Netherlands → Netherlands
Japan vs Mexico → Mexico
Morocco vs Switzerland → Morocco
Germany vs Australia → Germany
Qatar vs Egypt → Egypt
Sweden vs Algeria → Sweden
Austria vs France → Austria
Panama vs Argentina → Argentina
Colombia vs Norway → Norway
Spain vs Saudi Arabia → Spain
Croatia vs England → England
Belgium vs Canada → Canada
Portugal vs Turkey → Portugal
Scotland vs Ecuador → Ecuador
Uruguay vs Paraguay → Uruguay
South Korea vs Senegal → South Korea

=== Round of 16 ===
Netherlands vs Mexico → Mexico
Morocco vs Germany → Germany
Egypt vs Sweden → Egypt
Austria vs Argentina → Austria
Norway vs Spain → Spain
England vs Canada → England
Portugal vs Ecuador → Ecuador
Uruguay vs South Korea → Uruguay

=== Round of 8 ===
Mexico vs Germany → Mexico
Egypt vs Austria → Austria
Spain vs England → Spain
Ecuador vs Uruguay → Uruguay

=== Round of 4 ===
Mexico vs Austria → Mexico
Spain vs Ur

In [34]:
markov_winner, bracket = simulate_world_cup(markov_df, 'rating', groups, 'Markov')

print_bracket(bracket)


--- World Cup Simulation (Markov) ---

🏆 Winner (Markov): Netherlands

=== Round of 32 ===
Paraguay vs France → Paraguay
Portugal vs Croatia → Croatia
Netherlands vs Ivory Coast → Netherlands
South Korea vs Brazil → South Korea
Spain vs Mexico → Mexico
Argentina vs Czech Republic → Argentina
Senegal vs Germany → Germany
Egypt vs England → Egypt
Algeria vs Norway → Norway
Iran vs Belgium → Belgium
Turkey vs Switzerland → Turkey
Scotland vs Colombia → Colombia
Japan vs Austria → Japan
Uruguay vs Qatar → Uruguay
Saudi Arabia vs Sweden → Sweden
Australia vs Morocco → Morocco

=== Round of 16 ===
Paraguay vs Croatia → Croatia
Netherlands vs South Korea → Netherlands
Mexico vs Argentina → Argentina
Germany vs Egypt → Egypt
Norway vs Belgium → Belgium
Turkey vs Colombia → Colombia
Japan vs Uruguay → Japan
Sweden vs Morocco → Sweden

=== Round of 8 ===
Croatia vs Netherlands → Netherlands
Argentina vs Egypt → Egypt
Belgium vs Colombia → Colombia
Japan vs Sweden → Japan

=== Round of 4 ===
Net

### Comparative Summary of Top Teams by Rating Method

In [35]:
comparative_top_teams = pd.DataFrame({
    'Massey Ratings': massey_df.head(8)['team'].tolist(),
    'Keener Ratings': keener_df.head(8)['team'].tolist(),
    'Markov Ratings': markov_df.head(8)['team'].tolist()
})

print(f"Massey Winner: {massey_winner}")
print(f"Keener Winner: {keener_winner}")
print(f"Markov Winner: {markov_winner}")

display(comparative_top_teams)

Massey Winner: Argentina
Keener Winner: Mexico
Markov Winner: Netherlands


,Massey Ratings,Keener Ratings,Markov Ratings
0,France,Spain,Spain
1,Spain,France,France
2,Brazil,Brazil,Brazil
3,Portugal,Croatia,Argentina
4,Argentina,Argentina,Netherlands
5,Belgium,Colombia,Switzerland
6,Netherlands,Netherlands,Japan
7,England,Portugal,Colombia


### Run Simulations with the General Function

### 6. Monte Carlo Simulation for World Cup Winner Probability

Instead of just a single simulation, we'll run the entire World Cup simulation multiple times (e.g., 1000 times) for each rating system. This will give us a more robust estimate of each team's probability of winning the tournament.

In [ ]:
num_simulations = 1000

def run_monte_carlo_simulation(ratings_df, rating_column_name, groups, simulation_name, num_simulations):
    winners = []
    # print(f"\n--- Running Monte Carlo Simulation ({simulation_name}) for {num_simulations} iterations ---")
    for _ in range(num_simulations):
        winner, _ = simulate_world_cup(ratings_df, rating_column_name, groups, simulation_name, initial_round_size=32)
        winners.append(winner)
    return winners

# Run Monte Carlo for Massey ratings
massey_monte_carlo_winners = run_monte_carlo_simulation(massey_df, 'massey', groups, 'Massey (Monte Carlo)', num_simulations)

# Run Monte Carlo for Keener ratings
keener_monte_carlo_winners = run_monte_carlo_simulation(keener_df, 'rating', groups, 'Keener (Monte Carlo)', num_simulations)

# Run Monte Carlo for Markov ratings
markov_monte_carlo_winners = run_monte_carlo_simulation(markov_df, 'rating', groups, 'Markov (Monte Carlo)', num_simulations)


### Monte Carlo Results: Most Probable Winners

In [37]:
from collections import Counter

def analyze_monte_carlo_results(winners_list, simulation_name):
    winner_counts = Counter(winners_list)
    total_simulations = len(winners_list)
    win_probabilities = {team: count / total_simulations for team, count in winner_counts.items()}

    most_probable_winner = max(win_probabilities, key=win_probabilities.get)
    max_probability = win_probabilities[most_probable_winner]

    print(f"\n--- Monte Carlo Results for {simulation_name} ---")
    print(f"Most probable winner: {most_probable_winner} with a probability of {max_probability:.2%}")

    # Display top 5 probable winners
    print("\nTop 8 Most Probable Winners:")
    for team, prob in sorted(win_probabilities.items(), key=lambda item: item[1], reverse=True)[:8]:
        print(f"{team}: {prob:.2%}")

# Analyze and print results for each rating system
analyze_monte_carlo_results(massey_monte_carlo_winners, 'Massey Ratings')
analyze_monte_carlo_results(keener_monte_carlo_winners, 'Keener Ratings')
analyze_monte_carlo_results(markov_monte_carlo_winners, 'Markov Ratings')



--- Monte Carlo Results for Massey Ratings ---
Most probable winner: Portugal with a probability of 11.90%

Top 8 Most Probable Winners:
Portugal: 11.90%
France: 11.70%
Spain: 10.60%
Brazil: 10.30%
Belgium: 10.30%
Argentina: 9.40%
Netherlands: 8.10%
England: 6.60%

--- Monte Carlo Results for Keener Ratings ---
Most probable winner: Spain with a probability of 9.40%

Top 8 Most Probable Winners:
Spain: 9.40%
France: 8.40%
Brazil: 7.90%
Croatia: 6.70%
Netherlands: 6.00%
Portugal: 5.90%
Argentina: 5.10%
Colombia: 5.00%

--- Monte Carlo Results for Markov Ratings ---
Most probable winner: Spain with a probability of 13.80%

Top 8 Most Probable Winners:
Spain: 13.80%
France: 12.70%
Brazil: 7.70%
Argentina: 6.50%
Japan: 6.10%
Colombia: 5.40%
Netherlands: 5.20%
Switzerland: 5.10%
